In [1]:
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, StandardScaler

In [2]:
TRAIN_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\train.csv"
TEST_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\test.csv"
SUBMISSION_data = r"C:\Users\hp\Desktop\Data_analysis-and-ML\sample_submission.csv"

train = pd.read_csv(TRAIN_data)
test = pd.read_csv(TEST_data)

print(train.shape, test.shape)
train.head()


KeyboardInterrupt



In [ ]:
train = train.drop_duplicates()
test_ID = test["Id"]

def features(df):
    df = df.copy()

    if "Id" in df.columns:
        df = df.drop(columns=["Id"])

    na_as_none_cols = [
        "Alley", "BsmtQual", "BsmtCond", "BsmtExposure",
        "BsmtFinType1", "BsmtFinType2", "FireplaceQu",
        "GarageType", "GarageFinish", "GarageQual", "GarageCond",
        "PoolQC", "Fence", "MiscFeature"
    ]

    for col in na_as_none_cols:
        if col in df.columns:
            df[col] = df[col].fillna("None")

    zero_fill_cols = [
        "GarageYrBlt", "GarageArea", "GarageCars",
        "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
        "BsmtFullBath", "BsmtHalfBath", "MasVnrArea"
    ]

    for col in zero_fill_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)
    
    if {"1stFlrSF", "2ndFlrSF", "TotalBsmtSF"}.issubset(df.columns):
        df["TotalSF"] = df["1stFlrSF"] + df["2ndFlrSF"] + df["TotalBsmtSF"]

    if {"FullBath", "HalfBath", "BsmtFullBath", "BsmtHalfBath"}.issubset(df.columns):
        df["TotalBath"] = (
            df["FullBath"]
            + 0.5 * df["HalfBath"]
            + df["BsmtFullBath"]
            + 0.5 * df["BsmtHalfBath"]
        )

    if {"YrSold", "YearBuilt"}.issubset(df.columns):
        df["HouseAge"] = df["YrSold"] - df["YearBuilt"]

    if {"YrSold", "YearRemodAdd"}.issubset(df.columns):
        df["RemodAge"] = df["YrSold"] - df["YearRemodAdd"]

    if {"GrLivArea", "LotArea"}.issubset(df.columns):
        df["LivLotRatio"] = df["GrLivArea"] / (df["LotArea"] + 1)

    if {"OverallQual", "OverallCond"}.issubset(df.columns):
        df["TotalHomeQuality"] = df["OverallQual"] * df["OverallCond"]

    if {"OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"}.issubset(df.columns):
        df["TotalPorchSF"] = (
            df["OpenPorchSF"]
            + df["EnclosedPorch"]
            + df["3SsnPorch"]
            + df["ScreenPorch"]
        )

    if "GarageArea" in df.columns:
        df["HasGarage"] = (df["GarageArea"] > 0).astype(int)

    if "TotalBsmtSF" in df.columns:
        df["HasBsmt"] = (df["TotalBsmtSF"] > 0).astype(int)

    if "Fireplaces" in df.columns:
        df["HasFireplace"] = (df["Fireplaces"] > 0).astype(int)

    if "PoolArea" in df.columns:
        df["HasPool"] = (df["PoolArea"] > 0).astype(int)

    if "2ndFlrSF" in df.columns:
        df["Has2ndFloor"] = (df["2ndFlrSF"] > 0).astype(int)

    if {"WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"}.issubset(df.columns):
        df["TotalOutdoorSF"] = (  
            df["WoodDeckSF"]
            + df["OpenPorchSF"]
            + df["EnclosedPorch"]
            + df["3SsnPorch"]
            + df["ScreenPorch"]
        )

    if {"TotRmsAbvGrd", "KitchenAbvGr"}.issubset(df.columns):
        df["TotalRooms"] = df["TotRmsAbvGrd"] + df["KitchenAbvGr"]  

    if {"GarageArea", "GarageCars"}.issubset(df.columns):
        df["GarageScore"] = df["GarageArea"] * df["GarageCars"]  

    if {"OverallQual", "GrLivArea"}.issubset(df.columns):
        df["QualArea"] = df["OverallQual"] * df["GrLivArea"]  

    if {"YearRemodAdd", "YearBuilt"}.issubset(df.columns):
        df["IsRemodeled"] = (df["YearRemodAdd"] > df["YearBuilt"]).astype(int)  

    if {"YrSold", "YearBuilt"}.issubset(df.columns):
        df["IsNewHouse"] = (df["YrSold"] == df["YearBuilt"]).astype(int)  

    if {"TotalBath", "TotRmsAbvGrd"}.issubset(df.columns):
        df["BathPerRoom"] = df["TotalBath"] / (df["TotRmsAbvGrd"] + 1)  

    if {"OverallQual", "TotalSF"}.issubset(df.columns):
        df["Qual_x_TotalSF"] = df["OverallQual"] * df["TotalSF"]  

    if {"HouseAge", "OverallQual"}.issubset(df.columns):
        df["Age_x_Qual"] = df["HouseAge"] * df["OverallQual"]  

    skewed_cols = ["LotArea", "GrLivArea", "TotalBsmtSF", "1stFlrSF", "MasVnrArea"]  
    for col in skewed_cols:  
        if col in df.columns:  
            df[col] = np.log1p(df[col])  

    if "OverallQual" in df.columns:
        df["OverallQual2"] = df["OverallQual"] ** 2  

    if "TotalSF" in df.columns:
        df["TotalSF2"] = df["TotalSF"] ** 2  

    return df


def group_rare_categories(train_df, test_df, min_count=10):  
    train_df = train_df.copy()  
    test_df = test_df.copy()  

    cat_cols = train_df.select_dtypes(include=["object"]).columns.tolist()  

    for col in cat_cols:  
        freq = train_df[col].value_counts()  
        rare_values = freq[freq < min_count].index  

        train_df[col] = train_df[col].replace(rare_values, "Rare")  
        test_df[col] = test_df[col].replace(rare_values, "Rare")  

    return train_df, test_df  

def log_transform_skewed(train_df, test_df, threshold=0.75):
    train_df = train_df.copy()
    test_df = test_df.copy()

    num_cols = train_df.select_dtypes(include=[np.number]).columns.tolist()

    skewness = train_df[num_cols].skew().sort_values(ascending=False)
    skewed_cols = skewness[skewness.abs() > threshold].index.tolist()

    for col in skewed_cols:
        if col in train_df.columns and col in test_df.columns:
            if train_df[col].min() >= 0 and test_df[col].min() >= 0:
                train_df[col] = np.log1p(train_df[col])
                test_df[col] = np.log1p(test_df[col])

    return train_df, test_df, skewed_cols

train_fe = features(train)
test_fe = features(test)

train_fe, test_fe = group_rare_categories(train_fe, test_fe)

train_fe, test_fe, skewed_cols = log_transform_skewed(train_fe, test_fe)

X = train_fe.drop(columns=["SalePrice"])
y = train_fe["SalePrice"]
X_test = test_fe.copy()

cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

In [ ]:
def make_preprocessor(X, scaler_type="robust"):
    cat_cols = X.select_dtypes(include=["object"]).columns.tolist()
    num_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

    if scaler_type == "robust":
        scaler = RobustScaler()
    elif scaler_type == "standard":
        scaler = StandardScaler()
    else:
        scaler = "passthrough"

    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", scaler)
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, num_cols),
        ("cat", categorical_pipeline, cat_cols)
    ])

    return preprocessor

In [ ]:
def cv_mae_original_scale(model, X, y, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []

    for fold, (tr_idx, va_idx) in enumerate(kf.split(X), start=1):
        X_train, X_valid = X.iloc[tr_idx], X.iloc[va_idx]
        y_train, y_valid = y.iloc[tr_idx], y.iloc[va_idx]

        m = clone(model)
        m.fit(X_train, np.log1p(y_train))

        preds = np.expm1(m.predict(X_valid))
        preds = np.clip(preds, 10000, None)

        mae = mean_absolute_error(y_valid, preds)
        scores.append(mae)

        print(f"Fold {fold}: {mae:,.2f}")

    print("Mean CV MAE:", f"{np.mean(scores):,.2f}")
    print("Std CV MAE: ", f"{np.std(scores):,.2f}")
    return np.mean(scores), np.std(scores)

In [ ]:
preprocessor = make_preprocessor(X, scaler_type="robust")

alphas = [0.0001, 0.0003, 0.0005, 0.001, 0.003, 0.01]
l1_ratios = [0.05, 0.1, 0.15, 0.2, 0.4, 0.6]

results = []

for alpha in alphas:
    for l1_ratio in l1_ratios:
        model = Pipeline([
            ("preprocessor", preprocessor),
            ("regressor", ElasticNet(
                alpha=alpha,
                l1_ratio=l1_ratio,
                max_iter=30000,
                random_state=42
            ))
        ])

        mean_mae, std_mae = cv_mae_original_scale(model, X, y)

        results.append({
            "alpha": alpha,
            "l1_ratio": l1_ratio,
            "mean_mae": mean_mae,
            "std_mae": std_mae
        })

results_df = pd.DataFrame(results).sort_values("mean_mae")
print(results_df.head(10))

Fold 1: 17,329.37
Fold 2: 16,291.11
Fold 3: 18,449.39
Fold 4: 15,297.14
Fold 5: 16,745.23
Mean CV MAE: 16,822.45
Std CV MAE:  1,050.59
Fold 1: 17,307.91
Fold 2: 16,238.38
Fold 3: 18,420.58
Fold 4: 15,216.34
Fold 5: 16,706.18
Mean CV MAE: 16,777.88
Std CV MAE:  1,069.02
Fold 1: 17,284.88
Fold 2: 16,185.96
Fold 3: 18,388.35
Fold 4: 15,143.65
Fold 5: 16,667.70
Mean CV MAE: 16,734.11
Std CV MAE:  1,083.91
Fold 1: 17,258.13
Fold 2: 16,136.44
Fold 3: 18,360.49
Fold 4: 15,069.36
Fold 5: 16,634.42
Mean CV MAE: 16,691.77
Std CV MAE:  1,099.93
Fold 1: 17,132.21
Fold 2: 15,946.08
Fold 3: 18,254.72
Fold 4: 14,807.86
Fold 5: 16,423.29
Mean CV MAE: 16,512.84
Std CV MAE:  1,153.60
Fold 1: 17,000.26
Fold 2: 15,785.57
Fold 3: 18,139.45
Fold 4: 14,544.82


c:\Users\hp\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.351e-01, tolerance: 1.541e-02
  model = cd_fast.enet_coordinate_descent(


Fold 5: 16,236.74
Mean CV MAE: 16,341.37
Std CV MAE:  1,201.22
Fold 1: 17,183.66
Fold 2: 16,080.80
Fold 3: 18,342.91
Fold 4: 14,929.47
Fold 5: 16,564.14
Mean CV MAE: 16,620.20
Std CV MAE:  1,134.71
Fold 1: 17,073.42
Fold 2: 15,935.89
Fold 3: 18,248.00
Fold 4: 14,762.09
Fold 5: 16,422.87
Mean CV MAE: 16,488.45
Std CV MAE:  1,160.01
Fold 1: 16,968.25
Fold 2: 15,825.05
Fold 3: 18,163.74
Fold 4: 14,595.44
Fold 5: 16,284.64
Mean CV MAE: 16,367.43
Std CV MAE:  1,185.64
Fold 1: 16,873.65
Fold 2: 15,730.36
Fold 3: 18,097.83
Fold 4: 14,432.56
Fold 5: 16,160.31
Mean CV MAE: 16,258.94
Std CV MAE:  1,215.25
Fold 1: 16,558.65
Fold 2: 15,446.73
Fold 3: 17,810.46
Fold 4: 14,016.57
Fold 5: 15,787.52
Mean CV MAE: 15,923.99
Std CV MAE:  1,252.70
Fold 1: 16,250.96
Fold 2: 15,207.14
Fold 3: 17,595.27
Fold 4: 13,697.30
Fold 5: 15,579.35
Mean CV MAE: 15,666.01
Std CV MAE:  1,277.35
Fold 1: 16,993.68
Fold 2: 15,900.39
Fold 3: 18,212.78
Fold 4: 14,692.46
Fold 5: 16,377.84
Mean CV MAE: 16,435.43
Std CV MAE:  1

In [ ]:
best_alpha = results_df.iloc[0]["alpha"]
best_l1_ratio = results_df.iloc[0]["l1_ratio"]

preprocessor = make_preprocessor(X, scaler_type="robust")

elastic_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", ElasticNet(
        alpha=best_alpha,
        l1_ratio=best_l1_ratio,
        max_iter=30000,
        random_state=42
    ))
])

ridge_alphas = [1.0, 3.0, 5.0, 10.0, 20.0, 30.0]
ridge_results = []

for alpha in ridge_alphas:
    preprocessor = make_preprocessor(X, scaler_type="robust")

    ridge_model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", Ridge(alpha=alpha))
    ])

    mean_mae, std_mae = cv_mae_original_scale(ridge_model, X, y)

    ridge_results.append({
        "alpha": alpha,
        "mean_mae": mean_mae,
        "std_mae": std_mae
    })
ridge_results_df = pd.DataFrame(ridge_results).sort_values("mean_mae")
print(ridge_results_df)

elastic_model.fit(X, np.log1p(y))

elastic_preds = np.expm1(elastic_model.predict(X_test))
elastic_preds = np.clip(elastic_preds, 10000, None)

best_ridge_alpha = ridge_results_df.iloc[0]["alpha"]

preprocessor = make_preprocessor(X, scaler_type="robust")

ridge_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=best_ridge_alpha))
])

ridge_model.fit(X, np.log1p(y))

ridge_preds = np.expm1(ridge_model.predict(X_test))
ridge_preds = np.clip(ridge_preds, 10000, None)

mean_mae, std_mae = cv_mae_original_scale(elastic_model, X, y)
print("Best ElasticNet CV MAE:", mean_mae)

final_preds = 0.6 * elastic_preds + 0.4 * ridge_preds

Fold 1: 16,933.34
Fold 2: 15,925.11
Fold 3: 18,255.71
Fold 4: 14,662.30
Fold 5: 16,407.39
Mean CV MAE: 16,436.77
Std CV MAE:  1,180.38
Fold 1: 16,359.81
Fold 2: 15,482.89
Fold 3: 17,940.96
Fold 4: 14,177.26
Fold 5: 15,997.02
Mean CV MAE: 15,991.59
Std CV MAE:  1,223.62
Fold 1: 16,053.72
Fold 2: 15,294.21
Fold 3: 17,745.29
Fold 4: 14,059.71
Fold 5: 15,862.56
Mean CV MAE: 15,803.10
Std CV MAE:  1,194.72
Fold 1: 15,726.90
Fold 2: 15,104.70
Fold 3: 17,441.53
Fold 4: 13,954.67
Fold 5: 15,776.60
Mean CV MAE: 15,600.88
Std CV MAE:  1,130.57
Fold 1: 15,667.32
Fold 2: 15,182.81
Fold 3: 17,129.73
Fold 4: 13,935.42
Fold 5: 15,897.17
Mean CV MAE: 15,562.49
Std CV MAE:  1,036.41
Fold 1: 15,786.38
Fold 2: 15,352.61
Fold 3: 16,977.56
Fold 4: 13,981.39
Fold 5: 16,023.65
Mean CV MAE: 15,624.32
Std CV MAE:  978.78
   alpha      mean_mae      std_mae
4   20.0  15562.490552  1036.413036
3   10.0  15600.880968  1130.568313
5   30.0  15624.317050   978.776649
2    5.0  15803.098114  1194.723755
1    3.0  15